# 📊 지수 분석 대시보드

이 노트북은 다양한 연구자 평가 지수 실험을 인터랙티브하게 실행할 수 있는 대시보드입니다.

> ⚠️ 먼저 ipywidgets 설치가 필요할 수 있습니다:
```bash
pip install ipywidgets
```

In [ ]:
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from index_evaluator import load_authors_data, evaluate_all_with_expert_index
from viz_utils import (
    plot_index_correlation_heatmap,
    print_top_k_authors,
    compute_top_k_overlap_matrix,
    plot_top_k_overlap_heatmap,
    plot_rank_shift,
    plot_unique_author_counts_by_k,
    plot_index_clustermap,
    analyze_extreme_rank_shift,
    INDEX_NAME_MAP
)

# 경로 설정
authors_file = "./gsc_data/authors.all"
data_dir = "./gsc_data/DATA/"
output_file = "all_index_results.csv"

authors_data = load_authors_data(authors_file)

if not os.path.exists(output_file):
    index_df = evaluate_all_with_expert_index(authors_data, data_dir, current_year=2013)
    index_df.to_csv(output_file, index=False)
else:
    index_df = pd.read_csv(output_file)

# 준비
index_df_short = index_df.rename(columns=INDEX_NAME_MAP)
index_columns = list(INDEX_NAME_MAP.values())


In [ ]:
def run_selected_experiment(selection):
    clear_output(wait=True)
    display(dropdown, button)
    
    if selection == "1. Pearson 상관 히트맵":
        plot_index_correlation_heatmap(index_df, method="pearson")
        
    elif selection == "2. Spearman 상관 히트맵":
        plot_index_correlation_heatmap(index_df, method="spearman")
        
    elif selection == "3. Top-K 저자 출력":
        print_top_k_authors(index_df_short, 10, index_columns)
        
    elif selection == "4. Top-K 겹침 히트맵":
        overlap = compute_top_k_overlap_matrix(index_df_short, index_columns, k=10)
        plot_top_k_overlap_heatmap(overlap)
        
    elif selection == "5. 지수별 고유 저자 수 (Top-K별 선그래프)":
        k_range = [5, 10, 15, 20, 25, 30]
        plot_unique_author_counts_by_k(index_df_short, index_columns, k_range)
        
    elif selection == "6. Clustermap (지수 유사도 기반)":
        overlap = compute_top_k_overlap_matrix(index_df_short, index_columns, k=10)
        plot_index_clustermap(overlap, k=10)
        
    elif selection == "7. expert 기준 랭킹 변화 시각화":
        plot_rank_shift(index_df_short, index_columns, base_index='expert', top_n=15)
        
    elif selection == "8. expert 기준 랭킹 급변 저자 분석":
        analyze_extreme_rank_shift(index_df_short, index_columns, base_index='expert', top_n=5)


In [ ]:
dropdown = widgets.Dropdown(
    options=[
        "1. Pearson 상관 히트맵",
        "2. Spearman 상관 히트맵",
        "3. Top-K 저자 출력",
        "4. Top-K 겹침 히트맵",
        "5. 지수별 고유 저자 수 (Top-K별 선그래프)",
        "6. Clustermap (지수 유사도 기반)",
        "7. expert 기준 랭킹 변화 시각화",
        "8. expert 기준 랭킹 급변 저자 분석"
    ],
    description="실험 선택:"
)

button = widgets.Button(description="실행")
button.on_click(lambda b: run_selected_experiment(dropdown.value))

display(dropdown, button)
